# 02 - Upstream QC: Guide RNA Assignment

This notebook performs quality control on guide RNA UMI counts for a combinatorial
CRISPR screen. It filters low-confidence guide assignments, classifies cells as
KO or control, fits a Poisson model to the guide multiplicity distribution, and
saves the filtered AnnData object.

## Setup

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import poisson

from config import (
    ALL_GUIDE_COLUMNS,
    ANNDATA_RAW,
    CONTROL_COLUMNS,
    GUIDE_UMI_FRAC_THRESHOLD,
    KO_COLUMNS,
    PROJECT_DIR,
)

sns.set_style("whitegrid")
%matplotlib inline

## Helper functions

In [ ]:
from typing import List, Optional, Tuple, Union


def plot_faceted_histograms(
    df: pd.DataFrame,
    columns: List[str],
    *,
    xlim: Tuple[float, float],
    title: str,
    exclude_zeros: bool = False,
    vline: Optional[float] = None,
    binwidth: Optional[float] = None,
    bins: int = 50,
    ncols: int = 4,
    ylabel: str = "Count",
    xlabel: str = "",
) -> None:
    """Draw faceted histograms, one per guide column, matching ggplot2 facet_wrap with free_y."""
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_2d(axes).flatten()

    for idx, col in enumerate(columns):
        ax = axes[idx]
        vals = df[col]
        if exclude_zeros:
            vals = vals[vals > 0]
        if len(vals) == 0:
            ax.set_title(col, fontsize=12)
            continue

        if binwidth is not None:
            bin_edges = np.arange(xlim[0], xlim[1] + binwidth, binwidth)
        else:
            bin_edges = bins

        ax.hist(vals, bins=bin_edges, edgecolor="black", facecolor="white", linewidth=0.5)
        ax.set_xlim(xlim)
        ax.set_title(col, fontsize=12)
        ax.tick_params(labelsize=10)
        if vline is not None:
            ax.axvline(vline, color="red", linestyle="-", linewidth=1)
        if xlabel:
            ax.set_xlabel(xlabel, fontsize=10)

    for idx in range(len(columns), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(title, fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

## Load data

In [ ]:
adata = ad.read_h5ad(ANNDATA_RAW)
os.chdir(PROJECT_DIR)
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")
adata

In [ ]:
guideMat = adata.obs[KO_COLUMNS + CONTROL_COLUMNS].copy()
guideMat.head()

## Raw guide UMI distributions

In [ ]:
plot_faceted_histograms(
    guideMat,
    KO_COLUMNS + CONTROL_COLUMNS,
    xlim=(0, 50),
    title="Raw guide UMI counts (zeros excluded)",
    exclude_zeros=True,
    binwidth=1,
    ncols=4,
    xlabel="Number of UMIs",
)

## Row-normalised UMI percentage distributions

In [ ]:
guideMatPerc = guideMat.div(guideMat.sum(axis=1), axis=0)
guideMatPerc.head()

In [ ]:
plot_faceted_histograms(
    guideMatPerc,
    KO_COLUMNS + CONTROL_COLUMNS,
    xlim=(0, 1.2),
    title="UMI percentage distributions (pre-filter)",
    exclude_zeros=True,
    binwidth=0.01,
    vline=GUIDE_UMI_FRAC_THRESHOLD,
    ncols=4,
    xlabel="UMI percentage",
)

## Filter guides below UMI fraction threshold

In [ ]:
low_mask = guideMatPerc < GUIDE_UMI_FRAC_THRESHOLD
guideMat[low_mask] = 0
guideMatPerc[low_mask] = 0

print(f"Zeroed {low_mask.sum().sum()} guide-cell entries below {GUIDE_UMI_FRAC_THRESHOLD}")

## Filtered guide UMI distributions

In [ ]:
plot_faceted_histograms(
    guideMat,
    KO_COLUMNS + CONTROL_COLUMNS,
    xlim=(0, 50),
    title="Filtered guide UMI counts (zeros excluded)",
    exclude_zeros=True,
    binwidth=1,
    ncols=4,
    xlabel="Number of UMIs",
)

## Filtered UMI percentage distributions

In [ ]:
plot_faceted_histograms(
    guideMatPerc,
    KO_COLUMNS + CONTROL_COLUMNS,
    xlim=(0, 1.2),
    title="UMI percentage distributions (post-filter)",
    exclude_zeros=True,
    binwidth=0.01,
    vline=GUIDE_UMI_FRAC_THRESHOLD,
    ncols=6,
    xlabel="Cellular UMI percentage",
)

## Binary guide assignment and cell classification

In [ ]:
guideMatPercAssign = (guideMatPerc > 0).astype(int)
guideMatPercAssign["NumKOGuides"] = guideMatPercAssign[KO_COLUMNS].sum(axis=1)
guideMatPercAssign["NumControlGuides"] = guideMatPercAssign[CONTROL_COLUMNS].sum(axis=1)

guideMatPercAssign[["NumKOGuides", "NumControlGuides"]].describe()

In [ ]:
controlCells = guideMatPercAssign[
    (guideMatPercAssign["NumKOGuides"] == 0)
    & (guideMatPercAssign["NumControlGuides"] > 0)
]
koCells = guideMatPercAssign[
    (guideMatPercAssign["NumKOGuides"] > 0)
    & (guideMatPercAssign["NumControlGuides"] == 0)
]

print(f"Control cells: {len(controlCells)}")
print(f"KO cells:      {len(koCells)}")

## Poisson fit to KO guide multiplicity

In [ ]:
ko_guide_counts = koCells["NumKOGuides"]
lam = ko_guide_counts.mean()

max_k = int(ko_guide_counts.max())
k_vals = np.arange(1, max_k + 1)

observed_frac = ko_guide_counts.value_counts(normalize=True).reindex(k_vals, fill_value=0)
poisson_pmf = poisson.pmf(k_vals, lam)
poisson_pmf = poisson_pmf / poisson_pmf.sum()  # renormalise over k >= 1

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(k_vals, observed_frac.values, color="steelblue", edgecolor="white", label="Observed")
ax.plot(k_vals, poisson_pmf, "ro-", linewidth=1.5, markersize=5, label=f"Poisson (\u03bb={lam:.2f})")
ax.set_xlabel("Number of KO guides per cell")
ax.set_ylabel("Fraction of KO cells")
ax.set_title("KO guide multiplicity vs. Poisson fit")
ax.legend()
fig.tight_layout()
plt.show()

## Filter AnnData to control + KO cells with < 3 KO guides

In [ ]:
keep_idx = controlCells.index.union(koCells.index)
adata = adata[adata.obs.index.isin(keep_idx)].copy()

adata.obs[KO_COLUMNS + CONTROL_COLUMNS] = guideMatPercAssign.loc[
    adata.obs.index, KO_COLUMNS + CONTROL_COLUMNS
]
adata.obs["NumKOGuides"] = guideMatPercAssign.loc[adata.obs.index, "NumKOGuides"]
adata.obs["NumControlGuides"] = guideMatPercAssign.loc[adata.obs.index, "NumControlGuides"]

print(f"Before NumKOGuides filter: {adata.n_obs} cells")
adata = adata[adata.obs["NumKOGuides"] < 3].copy()
print(f"After  NumKOGuides < 3:    {adata.n_obs} cells")

## Create guide label

In [ ]:
pCols = np.array(KO_COLUMNS + CONTROL_COLUMNS)
adata.obs["Guide"] = adata.obs[pCols].apply(
    lambda x: "-".join(pCols[np.where(x == 1)]), axis=1
)

print(f"Unique guide labels: {adata.obs['Guide'].nunique()}")
adata.obs["Guide"].value_counts().head(10)

## Guide cell counts

In [ ]:
guide_counts = adata.obs["Guide"].value_counts()

fig, ax = plt.subplots(figsize=(40, 20))
ax.bar(range(len(guide_counts)), guide_counts.values, color="steelblue")
ax.set_xticks(range(len(guide_counts)))
ax.set_xticklabels(guide_counts.index, rotation=90, fontsize=25)
ax.tick_params(axis="y", labelsize=35)
ax.set_yticks(np.arange(0, guide_counts.max() + 50, 50))
ax.set_xlabel("Guide names", fontsize=45)
ax.set_ylabel("Number of cells", fontsize=45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

## Save filtered AnnData

In [ ]:
adata.write(ANNDATA_RAW)
print(f"Saved filtered AnnData ({adata.n_obs} cells) to {ANNDATA_RAW}")

In [ ]:
adata

In [ ]:
ANNDATA_RAW